# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sami-shah-210504/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring (provisional).**
I chose this lane because in notebook 02 a hand-written rule and a small decision tree ranked pages differently, and the winner changed with the validation setup. I want to understand which ranking an editor can trust when deciding what to review first. I may confirm or change this lane by the end of Week 4.

In [6]:
import os, subprocess, pandas as pd, numpy as np
CSV = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(CSV):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
df = pd.read_csv(CSV)
print("Rows x columns:", df.shape)

Rows x columns: (30000, 44)


## 2. The question: decision, action, cost of a wrong call

- **Decision:** which pages should an editor review first for a refresh?
- **Who acts:** a content editor with limited time, who checks the top 50 pages and chooses refresh, protect or monitor.
- **Unit of analysis:** one pseudonymized content item (a page).
- **Output:** a ranked review queue with reason codes (for example, stale and visible).
- **Metric, chosen before modelling:** Precision@50, compared with the base rate of declining pages.
- **Target (a proxy):** `trend_direction == "down"`, a bucket from the current window, not a future outcome.
- **Cost of a wrong call:** editor hours go to pages that were not really declining, and a truly declining page ranked low goes unreviewed.
- **Why data/ML can help:** there are too many pages to review by hand, and one simple rule can't weigh several signals. My notebook 02 run showed a simple rule was strong at the very top of the list but weaker deeper down, so a learned ranking is worth testing.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
y = df["trend_direction"].str.lower().eq("down").astype(int).values
score = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int) * df["impressions_90d"]
order = np.argsort(-score.values)
print("Base rate declining:", round(y.mean(), 3))
print("Hand-rule Precision@50:", round(y[order[:50]].mean(), 3))

Base rate declining: 0.542
Hand-rule Precision@50: 0.68


## 3. Quick look at the data (2-3 real numbers)

Hand rule works at the top but covers very little: only 17 pages (0.06% of 30,000) are both stale and visible, and 16 of those 17 (94.1%) are labelled declining. That is strong at the very top of a queue, but with only 17 pages the rate is directional, not precise. It also means a review queue of 50 has 33 slots the rule can’t rank: those pages are tied at zero. If I subtract the 16 hits, the remaining 33 slots contain 18 declining pages (about 55%), close to the 54.2% base rate. So the rule’s Precision@50 of 0.68 comes almost entirely from its first 17 pages, and the rest is no better than chance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale   = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
declining = pd.Series(y, index=df.index).astype(bool)
top10_share = df["impressions_90d"].nlargest(len(df)//10).sum() / df["impressions_90d"].sum()

print("Clients:", df["client_id"].nunique(), "(guide says 32)")
print("avg_position == 0 ('no data'):", int((df["avg_position"] == 0).sum()), "(guide says 1,205)")
print("Share stale AND visible:", round((stale & visible).mean(), 3))
print("Declining rate among stale AND visible:", round(declining[stale & visible].mean(), 3))
print("Share of impressions held by top 10% of pages:", round(top10_share, 3))
print("Pages stale AND visible:", int((stale & visible).sum()))

Clients: 32 (guide says 32)
avg_position == 0 ('no data'): 1205 (guide says 1,205)
Share stale AND visible: 0.001
Declining rate among stale AND visible: 0.941
Share of impressions held by top 10% of pages: 0.702
Pages stale AND visible: 17


## 4. Careful words: what I can and can't claim

**I can say:** what I *observed* in this anonymized 30,000-row sample, and that a ranking is *directional decision support*.
**I can't say:** that a refresh *causes* recovery, that I know how Google ranks pages, or that results hold on the full warehouse.
**Data cautions:** the label is a proxy. `trend_direction` and `trend_pct` are never features, because the label is built from them. IDs are pseudonyms used only for grouping. Rate columns are percentages (ctr 0.76 means 0.76%).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
banned = {"trend_direction", "trend_pct", "content_id", "client_id"}
print("Banned columns in my feature list:", banned & set(features) or "none")


Banned columns in my feature list: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.